In [ ]:
# No GPU-heavy model downloads needed — all experiments run on CPU with numpy/torch
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from IPython.display import HTML, display

import torch

np.random.seed(42)
torch.manual_seed(42)

os.makedirs('outputs', exist_ok=True)

MASAI_RED = '#ED1C24'
BLUE      = '#1565C0'
GREEN     = '#2E7D32'
ORANGE    = '#E65100'
PURPLE    = '#7C4DFF'
GREY      = '#95A5A6'

print(f'PyTorch {torch.__version__}  |  NumPy {np.__version__}')
print('Setup complete.')


In [ ]:
from IPython.display import HTML, display

_BOX_STYLES = {
    "definition": ("#448aff", "#e3f2fd", "#1565c0"),
    "tip":        ("#00c853", "#e8f5e9", "#2e7d32"),
    "warning":    ("#ff9100", "#fff3e0", "#e65100"),
    "danger":     ("#ff1744", "#fce4ec", "#c62828"),
    "math":       ("#7c4dff", "#ede7f6", "#4527a0"),
    "output":     ("#00b8d4", "#e0f7fa", "#006064"),
    "industry":   ("#009688", "#e0f2f1", "#004d40"),
}

def box(kind, title, content):
    border, bg, title_clr = _BOX_STYLES[kind]
    html = (
        '<div style="margin:12px 0;padding:12px 16px;'
        'border-left:4px solid ' + border + ';'
        'background-color:' + bg + ';border-radius:4px;">'
        '<strong style="color:' + title_clr + ';">' + title + '</strong>'
        '<br>' + content + '</div>'
    )
    display(HTML(html))


In [ ]:
# Context window evolution across major LLMs
models = {
    'GPT-2 (2019)':          2_048,
    'GPT-3 (2020)':          4_096,
    'GPT-4 launch (2023)':   8_192,
    'Claude 2 (2023)':      100_000,
    'GPT-4 Turbo (2023)':   128_000,
    'Gemini 1.5 Pro (2024)':1_000_000,
    'Claude 3.5 (2024)':    200_000,
}

print("Context window evolution\n")
print(f"{'Model':<30} {'Tokens':>10}  {'~Pages of text':>15}  {'~Hours of speech':>18}")
print('-' * 78)
for name, tokens in models.items():
    pages   = tokens / 500        # ~500 tokens per page
    hours   = tokens / 8_000      # ~8000 tokens per hour of transcribed speech
    print(f"  {name:<28} {tokens:>10,}  {pages:>14.0f}  {hours:>17.1f}")

print()
print("Rule of thumb: 1 page ≈ 500 tokens  |  1 hour of speech ≈ 8,000 tokens")
print("              1 novel ≈ 100,000 tokens  |  full codebase ≈ varies widely")


In [ ]:
# Visualise: attention memory cost vs sequence length for different precisions
seq_lengths = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 128000])
n_layers = 32  # typical 7B model

precisions = {
    'FP32 (4 bytes)': 4,
    'FP16/BF16 (2 bytes)': 2,
    'INT8 (1 byte)': 1,
}

fig, ax = plt.subplots(figsize=(11, 6))
colors = [ORANGE, BLUE, GREEN]

for (label, bpv), color in zip(precisions.items(), colors):
    mem_gb = (seq_lengths ** 2 * bpv * n_layers) / (1024 ** 3)
    ax.plot(seq_lengths / 1000, mem_gb, marker='o', linewidth=2.5,
            color=color, label=label, markersize=6)

# Reference lines for common GPUs
gpu_vram = {'T4 (16 GB)': 16, 'A100 (40 GB)': 40, 'A100 (80 GB)': 80}
for gpu_name, vram in gpu_vram.items():
    ax.axhline(vram, linestyle='--', color=GREY, lw=1.2, alpha=0.7)
    ax.text(seq_lengths[-1] / 1000 + 0.5, vram + 0.5, gpu_name, fontsize=8, color=GREY)

ax.set_xlabel('Sequence Length (thousands of tokens)', fontsize=12)
ax.set_ylabel('Attention Memory (GB) — 32-layer model', fontsize=12)
ax.set_title('Attention Matrix Memory vs Sequence Length\n(32 layers — does NOT include model weights or KV cache)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_facecolor('#fafafa')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f} GB'))
plt.tight_layout()
plt.savefig('outputs/context_window_memory.png', dpi=150, bbox_inches='tight')
plt.show()

box("output", "Reading this chart",
    "The Y-axis is log-scale — memory grows quadratically with sequence length. "
    "At 128K tokens in FP32, attention matrices alone exceed 80 GB — more than an A100. "
    "Flash Attention sidesteps this by never materialising the full matrix, "
    "making long-context models practical at the cost of slightly more compute.")


In [ ]:
# Demonstrate the silent truncation problem
def simulate_context_truncation(text_tokens, context_window):
    """Show what happens when input exceeds context window."""
    if len(text_tokens) <= context_window:
        return text_tokens, 0, 'OK'
    truncated = text_tokens[:context_window]
    dropped   = len(text_tokens) - context_window
    pct_lost  = 100 * dropped / len(text_tokens)
    return truncated, dropped, f'TRUNCATED ({pct_lost:.1f}% dropped)'

# Simulate different document types
docs = {
    'Short tweet':            150,
    'News article':         1_200,
    '10-page report':       5_000,
    '50-page PDF':         25_000,
    '200-page book chapter':100_000,
    'Full Python codebase': 80_000,
}
context_windows = [4_096, 8_192, 32_768, 128_000]

print("Silent truncation matrix\n")
header = f"{'Document':<28}" + "".join(f"{'CW='+str(cw//1000)+'K':>14}" for cw in context_windows)
print(header)
print('-' * (28 + 14 * len(context_windows)))

for doc_name, doc_tokens in docs.items():
    row = f"  {doc_name:<26}"
    for cw in context_windows:
        _, dropped, status = simulate_context_truncation(list(range(doc_tokens)), cw)
        if dropped == 0:
            row += f"{'✓':>14}"
        else:
            pct = 100 * dropped / doc_tokens
            row += f"{f'−{pct:.0f}%':>14}"
    print(row)

print()
print("✓ = fits entirely  |  −X% = percentage of document silently DROPPED")
print()
print("Key insight: a 50-page PDF at CW=4K loses 84% of its content — silently.")

In [ ]:
# Visualise: naive vs cached attention computation cost
np.random.seed(7)

max_seq = 200
steps = np.arange(1, max_seq + 1)

# Naive: at each step t, do t attention computations (recompute all keys/values)
naive_ops = steps ** 2 / 2   # triangular sum ≈ t²/2

# Cached: at each step t, do only 1 new key/value computation (O(1) per step)
cached_ops = steps            # just the new token each time

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ops per step
ax = axes[0]
ax.plot(steps, steps,        color=MASAI_RED, lw=2.5, label='Naive: O(t) ops/step')
ax.plot(steps, np.ones_like(steps) * 1, color=BLUE, lw=2.5, label='Cached: O(1) ops/step')
ax.fill_between(steps, 1, steps, alpha=0.15, color=MASAI_RED)
ax.set_xlabel('Generation step t', fontsize=11)
ax.set_ylabel('New K/V computations at step t', fontsize=11)
ax.set_title('Compute per Generation Step', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.set_facecolor('#fafafa')

# Right: cumulative ops
ax = axes[1]
ax.plot(steps, naive_ops,  color=MASAI_RED, lw=2.5, label='Naive: O(n²) total')
ax.plot(steps, cached_ops, color=BLUE, lw=2.5, label='Cached: O(n) total')
ax.fill_between(steps, cached_ops, naive_ops, alpha=0.15, color=MASAI_RED)
ax.set_xlabel('Total tokens generated (n)', fontsize=11)
ax.set_ylabel('Cumulative K/V computation cost', fontsize=11)
ax.set_title('Cumulative Compute Cost', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.set_facecolor('#fafafa')

speedup_at_200 = naive_ops[-1] / cached_ops[-1]
ax.text(160, naive_ops[-1] * 0.6,
        f'{speedup_at_200:.0f}×\nspeedup\nat n={max_seq}',
        fontsize=11, color=MASAI_RED, fontweight='bold', ha='center')

plt.suptitle('KV Cache: O(n²) → O(n) Inference Cost Reduction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/kv_cache_speedup.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
box("math", "KV cache memory formula",
    "For a model with <i>L</i> layers, <i>H</i> heads, head-dimension <i>d_head</i>, "
    "generating a sequence of <i>S</i> tokens at precision <i>B</i> bytes:<br><br>"
    "<code>KV cache bytes = 2 × L × H × d_head × S × B</code><br><br>"
    "Factor of 2 = one K tensor + one V tensor per layer.<br><br>"
    "Example — LLaMA-2 7B (L=32, H=32, d_head=128) at BF16 (B=2) for S=4096 tokens:<br>"
    "  2 × 32 × 32 × 128 × 4096 × 2 bytes = 2.15 GB — just for the KV cache!<br><br>"
    "This is why serving a batch of 10 users each with a 4K context needs "
    "21.5 GB for the cache alone, on top of the 14 GB of model weights.")


In [ ]:
# KV cache memory calculator for popular open-source models
def kv_cache_gb(n_layers, n_heads, d_head, seq_len, bytes_per_value=2):
    """Calculate KV cache size in GB."""
    return (2 * n_layers * n_heads * d_head * seq_len * bytes_per_value) / (1024 ** 3)

# Model configs: (name, n_layers, n_heads, d_head)
model_configs = {
    'LLaMA-2 7B':       (32, 32, 128),
    'LLaMA-2 13B':      (40, 40, 128),
    'LLaMA-2 70B':      (80, 64, 128),
    'Mistral 7B':       (32, 32, 128),
    'GPT-3 175B':       (96, 96, 128),
    'Falcon 40B':       (60, 64, 64),
}

seq_lengths_cache = [1024, 4096, 16384, 32768, 128000]

print("KV Cache Memory (GB) at BF16 — per single request\n")
header = f"{'Model':<22}" + "".join(f"{'S='+str(s//1000)+'K':>10}" for s in seq_lengths_cache)
print(header)
print('-' * (22 + 10 * len(seq_lengths_cache)))

for model_name, (nl, nh, dh) in model_configs.items():
    row = f"  {model_name:<20}"
    for s in seq_lengths_cache:
        gb = kv_cache_gb(nl, nh, dh, s)
        row += f"{gb:>9.2f}G"
    print(row)

print()
print("Note: multiply by batch_size to get total KV cache for concurrent users.")
print("  e.g. 10 users × LLaMA-2 7B at S=4K = 10 × 0.54 GB = 5.4 GB KV cache")


In [ ]:
# Plot KV cache growth for LLaMA-2 7B — single request vs batch
seq_range = np.arange(256, 32769, 256)
batch_sizes = [1, 4, 8, 16, 32]

fig, ax = plt.subplots(figsize=(11, 6))
colors_batch = [BLUE, GREEN, ORANGE, PURPLE, MASAI_RED]

for bs, color in zip(batch_sizes, colors_batch):
    cache_gb = [kv_cache_gb(32, 32, 128, s) * bs for s in seq_range]
    ax.plot(seq_range / 1000, cache_gb, lw=2, color=color, label=f'Batch size = {bs}')

# GPU reference lines
for gpu_name, vram, ls in [('T4 16GB', 16, ':'), ('A100 40GB', 40, '--'), ('A100 80GB', 80, '-.')]:
    ax.axhline(vram, linestyle=ls, color=GREY, lw=1.5, alpha=0.8)
    ax.text(seq_range[-1] / 1000 + 0.3, vram + 0.5, gpu_name, fontsize=9, color=GREY)

ax.set_xlabel('Sequence Length (K tokens)', fontsize=12)
ax.set_ylabel('KV Cache Memory (GB)', fontsize=12)
ax.set_title('LLaMA-2 7B — KV Cache Memory vs Sequence Length\n(does NOT include 14 GB model weights)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.savefig('outputs/kv_cache_memory.png', dpi=150, bbox_inches='tight')
plt.show()

box("warning", "OOM danger: the KV cache is not free",
    "Engineers frequently forget the KV cache when estimating GPU requirements. "
    "A common mistake: load a 7B model on a 16 GB GPU (14 GB weights = 2 GB free), "
    "then serve a 4K-token conversation with batch_size=4 → 4 × 0.54 GB = 2.16 GB KV cache. "
    "The GPU OOMs mid-generation with a cryptic CUDA out-of-memory error. "
    "Always budget: <code>GPU VRAM ≥ model_weights + kv_cache + activations + overhead</code>")


In [ ]:
# Model size estimator from architecture parameters
def estimate_transformer_params(vocab_size, d_model, n_layers, n_heads,
                                  d_ff=None, max_seq_len=4096):
    """Approximate parameter count for a decoder-only Transformer (GPT-style)."""
    if d_ff is None:
        d_ff = 4 * d_model  # standard FFN expansion ratio

    # Embedding layers
    embed_params = vocab_size * d_model           # token embedding
    pos_params   = max_seq_len * d_model          # positional embedding (learned)

    # Per-layer parameters
    # Attention: Q, K, V, O projections
    attn_params = 4 * d_model * d_model
    # Feed-forward: two linear layers
    ffn_params  = 2 * d_model * d_ff
    # Layer norms: 2 × 2 × d_model (pre-attn + pre-ffn, each has weight + bias)
    ln_params   = 4 * d_model

    per_layer = attn_params + ffn_params + ln_params
    total     = embed_params + pos_params + n_layers * per_layer

    return {
        'embed':     embed_params,
        'pos_embed': pos_params,
        'per_layer': per_layer,
        'total':     total,
    }

# Common LLM architectures
architectures = {
    'GPT-2 Small':   dict(vocab_size=50257, d_model=768,   n_layers=12,  n_heads=12),
    'GPT-2 Large':   dict(vocab_size=50257, d_model=1280,  n_layers=36,  n_heads=20),
    'GPT-3 6.7B':    dict(vocab_size=50257, d_model=4096,  n_layers=32,  n_heads=32),
    'LLaMA-2 7B':    dict(vocab_size=32000, d_model=4096,  n_layers=32,  n_heads=32),
    'LLaMA-2 13B':   dict(vocab_size=32000, d_model=5120,  n_layers=40,  n_heads=40),
    'LLaMA-2 70B':   dict(vocab_size=32000, d_model=8192,  n_layers=80,  n_heads=64),
    'GPT-3 175B':    dict(vocab_size=50257, d_model=12288, n_layers=96,  n_heads=96),
}

print("Parameter count and memory footprint by precision\n")
print(f"{'Model':<18} {'Params (B)':>10}  {'FP32':>8}  {'FP16':>8}  {'INT8':>8}  {'INT4':>8}")
print('-' * 72)

for name, cfg in architectures.items():
    stats    = estimate_transformer_params(**cfg)
    n_params = stats['total']
    n_b      = n_params / 1e9
    fp32_gb  = n_params * 4  / 1024**3
    fp16_gb  = n_params * 2  / 1024**3
    int8_gb  = n_params * 1  / 1024**3
    int4_gb  = n_params * 0.5 / 1024**3
    print(f"  {name:<16} {n_b:>10.2f}  {fp32_gb:>7.1f}G  {fp16_gb:>7.1f}G  {int8_gb:>7.1f}G  {int4_gb:>7.1f}G")

print()
print("Memory = parameters × bytes_per_param (+ ~20% overhead for activations etc.)")


In [ ]:
# Visual: GPU memory required by model + KV cache for a typical serving scenario
models_viz = {
    'GPT-2 Small\n(124M)':  (0.124, 12, 12, 64),
    'LLaMA-2 7B\n(7B)':    (7.0,   32, 32, 128),
    'LLaMA-2 13B\n(13B)':  (13.0,  40, 40, 128),
    'LLaMA-2 70B\n(70B)':  (70.0,  80, 64, 128),
}
precisions = {'FP16': 2, 'INT8': 1, 'INT4': 0.5}
context_eval = 4096
batch_eval   = 4

fig, axes = plt.subplots(1, len(models_viz), figsize=(14, 6), sharey=False)

for ax, (model_label, (params_b, nl, nh, dh)) in zip(axes, models_viz.items()):
    weights_gb = {p: params_b * bpv for p, bpv in precisions.items()}
    kvc_gb     = {p: kv_cache_gb(nl, nh, dh, context_eval, bpv) * batch_eval
                  for p, bpv in precisions.items()}

    x      = np.arange(len(precisions))
    wgb    = list(weights_gb.values())
    kgb    = list(kvc_gb.values())

    bars1 = ax.bar(x, wgb, color=BLUE,    label='Model weights', width=0.5)
    bars2 = ax.bar(x, kgb, bottom=wgb, color=ORANGE, label=f'KV cache (bs={batch_eval}, S={context_eval//1024}K)', width=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(list(precisions.keys()), fontsize=10)
    ax.set_title(model_label, fontsize=10, fontweight='bold')
    ax.set_ylabel('GPU Memory (GB)' if ax == axes[0] else '', fontsize=10)
    ax.set_facecolor('#fafafa')

    for gpu_vram, ls in [(16, ':'), (40, '--'), (80, '-.')]:
        ax.axhline(gpu_vram, linestyle=ls, color=GREY, lw=1.0, alpha=0.7)

handles = [
    mpatches.Patch(color=BLUE, label='Model weights'),
    mpatches.Patch(color=ORANGE, label=f'KV cache (bs={batch_eval}, S={context_eval//1024}K)'),
]
axes[-1].legend(handles=handles, fontsize=9, loc='upper right')

plt.suptitle(f'GPU Memory = Weights + KV Cache\n(dashed lines: T4 16GB, A100 40GB, A100 80GB)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/model_memory_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Step-by-step INT8 quantisation of a weight matrix
np.random.seed(42)

# Simulate a weight matrix from a real model (values typically small, near-Gaussian)
W_fp32 = np.random.randn(8, 8).astype(np.float32) * 0.3

# INT8 symmetric quantisation
w_max = np.abs(W_fp32).max()
scale = w_max / 127.0                     # scale factor (stored as FP32)
W_int8 = np.round(W_fp32 / scale).astype(np.int8)   # quantised
W_dequant = W_int8.astype(np.float32) * scale        # dequantised for comparison

# Quantisation error
error = W_fp32 - W_dequant
max_err = np.abs(error).max()
mean_err = np.abs(error).mean()

print("INT8 Symmetric Quantisation — Step by Step\n")
print(f"  Original range : [{W_fp32.min():.4f}, {W_fp32.max():.4f}]")
print(f"  Scale factor   : {scale:.6f}  (stored as FP32, shared per tensor)")
print(f"  INT8 range     : [-128, 127]  (using [-127, 127] for symmetric)")
print()
print(f"  Sample original  : {W_fp32[0, :4].tolist()}")
print(f"  After quant INT8 : {W_int8[0, :4].tolist()}")
print(f"  After dequant    : {W_dequant[0, :4].round(4).tolist()}")
print()
print(f"  Max quantisation error  : {max_err:.6f}")
print(f"  Mean quantisation error : {mean_err:.6f}")
print()
print(f"  Memory: FP32 = {W_fp32.nbytes} bytes → INT8 = {W_int8.nbytes} bytes  ({W_fp32.nbytes//W_int8.nbytes}× reduction)")


In [ ]:
# Visualise quantisation: compare FP32, INT8, INT4 representations
np.random.seed(0)
x_vals = np.linspace(-1, 1, 500)
weights_1d = np.random.randn(1000).astype(np.float32) * 0.35

def quantise_fp(weights, bits):
    """Uniform symmetric quantisation to `bits` bits."""
    n_levels = 2 ** (bits - 1) - 1
    scale    = np.abs(weights).max() / n_levels
    q        = np.round(weights / scale).clip(-n_levels, n_levels).astype(np.float32)
    return q * scale  # dequantised

w_fp32 = weights_1d
w_int8 = quantise_fp(weights_1d, 8)
w_int4 = quantise_fp(weights_1d, 4)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
configs = [
    (w_fp32, 'FP32 (original)', BLUE,      'Infinite resolution\n≈32K distinct values'),
    (w_int8, 'INT8 quantised',  GREEN,     '256 distinct levels\n~0.5% typical error'),
    (w_int4, 'INT4 quantised',  MASAI_RED, '16 distinct levels\n~1-3% typical error'),
]
for ax, (w, title, color, note) in zip(axes, configs):
    ax.hist(w, bins=80, color=color, alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.set_title(f'{title}\n{note}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Weight value', fontsize=10)
    ax.set_facecolor('#fafafa')
    unique = len(np.unique(np.round(w, 5)))
    ax.text(0.97, 0.95, f'{unique} unique values',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.8))

axes[0].set_ylabel('Count', fontsize=10)
plt.suptitle('Weight Distribution: FP32 vs INT8 vs INT4 Quantisation\n(1000 sampled weights)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/quantisation_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

box("output", "Reading quantisation histograms",
    "FP32 weights have a smooth continuous distribution (nearly Gaussian). "
    "INT8 still looks smooth — 256 levels are fine-grained enough that the histogram bars are narrow. "
    "INT4 shows clear quantisation artefacts: weights cluster at 16 discrete values (visible as tall spikes). "
    "The broader the weight distribution, the more accuracy is lost — this is why "
    "outlier-aware quantisation methods (AWQ, SmoothQuant) pre-scale outlier channels before quantising.")


In [ ]:
# Accuracy-memory trade-off for LLaMA-2 7B (typical benchmark numbers from literature)
precision_data = {
    'FP32':        {'memory_gb': 28.0, 'mmlu_acc': 45.3, 'hellaswag': 77.1},
    'FP16/BF16':   {'memory_gb': 14.0, 'mmlu_acc': 45.3, 'hellaswag': 77.1},
    'INT8 (LLM.int8)': {'memory_gb': 7.0,  'mmlu_acc': 44.8, 'hellaswag': 76.8},
    'NF4 (QLoRA)': {'memory_gb': 3.9,  'mmlu_acc': 44.1, 'hellaswag': 76.4},
    'INT4 (GPTQ)': {'memory_gb': 3.5,  'mmlu_acc': 43.5, 'hellaswag': 75.9},
}

labels   = list(precision_data.keys())
mem_gb   = [v['memory_gb'] for v in precision_data.values()]
mmlu_acc = [v['mmlu_acc']  for v in precision_data.values()]
hella    = [v['hellaswag'] for v in precision_data.values()]

x = np.arange(len(labels))
width = 0.38

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: memory
ax = axes[0]
bars = ax.bar(x, mem_gb, color=[BLUE, BLUE, GREEN, ORANGE, MASAI_RED],
              width=0.55, edgecolor='white', linewidth=1.2)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Model Memory (GB)', fontsize=11)
ax.set_title('LLaMA-2 7B — Memory by Precision', fontsize=11, fontweight='bold')
ax.set_facecolor('#fafafa')
for bar, val in zip(bars, mem_gb):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f'{val:.1f} GB',
            ha='center', fontsize=9, fontweight='bold')

# Right: accuracy
ax = axes[1]
ax.plot(x, mmlu_acc, marker='o', color=BLUE, lw=2.5, markersize=8, label='MMLU (5-shot)')
ax.plot(x, hella,    marker='s', color=GREEN, lw=2.5, markersize=8, label='HellaSwag (0-shot)')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('LLaMA-2 7B — Benchmark Accuracy by Precision', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.set_facecolor('#fafafa')
ax.set_ylim(40, 82)

plt.suptitle('Quantisation Trade-off: Memory vs Accuracy (LLaMA-2 7B)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/quantisation_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

box("tip", "Practical quantisation choice guide",
    "<b>FP16/BF16</b>: default for fine-tuning and high-accuracy inference. Use this unless memory-constrained.<br>"
    "<b>INT8 (bitsandbytes LLM.int8())</b>: ~2× savings, <1% accuracy loss. "
    "Safe for most tasks; enable with <code>load_in_8bit=True</code> in HF Transformers.<br>"
    "<b>NF4 / INT4</b>: ~4× savings, 1-3% accuracy loss. "
    "Use for fitting large models on consumer GPUs (e.g., 70B on 2× A100-40GB). "
    "Enable with <code>load_in_4bit=True</code> + <code>bnb_4bit_quant_type='nf4'</code>.")


In [ ]:
def max_batch_size(gpu_vram_gb, model_params_b, bytes_per_param,
                    n_layers, n_heads, d_head, seq_len, overhead_factor=1.15):
    """
    Estimate the maximum batch size for a given GPU and model config.
    overhead_factor: multiplier on model weights to account for CUDA overhead.
    """
    weights_gb  = model_params_b * bytes_per_param * overhead_factor
    vram_free   = gpu_vram_gb - weights_gb
    if vram_free <= 0:
        return 0, weights_gb, 0
    kvc_per_req = kv_cache_gb(n_layers, n_heads, d_head, seq_len, bytes_per_param)
    max_bs      = max(0, int(vram_free / kvc_per_req))
    return max_bs, weights_gb, kvc_per_req

# GPU options
gpus = {
    'T4 (16 GB)':    16,
    'A10G (24 GB)':  24,
    'A100 (40 GB)':  40,
    'A100 (80 GB)':  80,
    'H100 (80 GB)':  80,
}

# Model options
models_bs = {
    'LLaMA-2 7B  FP16': (7.0, 32, 32, 128, 2),
    'LLaMA-2 7B  INT8': (7.0, 32, 32, 128, 1),
    'LLaMA-2 7B  INT4': (7.0, 32, 32, 128, 1),  # weights 0.5, but let's use 1 for KV
    'LLaMA-2 13B FP16': (13.0, 40, 40, 128, 2),
    'LLaMA-2 70B FP16': (70.0, 80, 64, 128, 2),
    'LLaMA-2 70B INT4': (70.0, 80, 64, 128, 1),
}
# Special INT4 weights override
weight_overrides = {
    'LLaMA-2 7B  INT4': 0.5,
    'LLaMA-2 70B INT4': 0.5,
}

seq_len_serving = 4096

print(f"Max concurrent requests (batch size) — context length = {seq_len_serving} tokens\n")
print(f"{'Model':<24}" + "".join(f"{gpu:>14}" for gpu in gpus.keys()))
print('-' * (24 + 14 * len(gpus)))

for model_label, (params_b, nl, nh, dh, bpv) in models_bs.items():
    bpv_weights = weight_overrides.get(model_label, bpv)
    row = f"  {model_label:<22}"
    for gpu_label, vram in gpus.items():
        bs, wgb, kvc = max_batch_size(vram, params_b, bpv_weights, nl, nh, dh, seq_len_serving)
        if wgb * 1.15 > vram:
            row += f"{'OOM':>14}"
        elif bs == 0:
            row += f"{'<1':>14}"
        else:
            row += f"{str(bs)+' req':>14}"
    print(row)

print()
print("OOM = model weights alone exceed GPU VRAM (can't even load the model)")


In [ ]:
# Heatmap: max batch size vs GPU size × context length for LLaMA-2 7B FP16
gpu_sizes = [16, 24, 40, 48, 80]
ctx_lengths = [1024, 2048, 4096, 8192, 16384, 32768]
model_w_gb = 7.0 * 2 * 1.15  # 7B FP16 + 15% overhead ≈ 16.1 GB

batch_matrix = np.zeros((len(gpu_sizes), len(ctx_lengths)))
for i, vram in enumerate(gpu_sizes):
    for j, sl in enumerate(ctx_lengths):
        free = vram - model_w_gb
        kvc  = kv_cache_gb(32, 32, 128, sl, 2)
        batch_matrix[i, j] = max(0, int(free / kvc)) if free > 0 else 0

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(batch_matrix, cmap='YlGn', aspect='auto',
               vmin=0, vmax=batch_matrix.max())
ax.set_xticks(range(len(ctx_lengths)))
ax.set_yticks(range(len(gpu_sizes)))
ax.set_xticklabels([f'{s//1024}K' for s in ctx_lengths], fontsize=11)
ax.set_yticklabels([f'{g} GB' for g in gpu_sizes], fontsize=11)
ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('GPU VRAM', fontsize=12)
ax.set_title('LLaMA-2 7B FP16 — Max Batch Size by GPU × Context Length\n(shaded = OOM for model weights alone)',
             fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Max concurrent requests')

for i in range(len(gpu_sizes)):
    for j in range(len(ctx_lengths)):
        val = int(batch_matrix[i, j])
        txt = str(val) if val > 0 else 'OOM'
        color = 'white' if val > batch_matrix.max() * 0.5 else 'black'
        ax.text(j, i, txt, ha='center', va='center', fontsize=11,
                fontweight='bold', color=color)

plt.tight_layout()
plt.savefig('outputs/batch_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

box("output", "Engineering insight from this heatmap",
    "Moving from a 16 GB T4 to a 24 GB A10G barely helps at long contexts "
    "because the model weights (16.1 GB) leave almost no room. "
    "A 40 GB A100 is the minimum practical choice for LLaMA-2 7B FP16 with any meaningful batching. "
    "Quantising to INT4 (≈4 GB weights) would allow 12+ concurrent requests on a T4 — "
    "a massive throughput improvement at a small accuracy cost.")
